# OGC: From Cluster Assumption to Graph Convolution

Semi-Supervised Node Classification on Cora: OGC (["From Cluster Assumption to Graph Convolution: Graph-based Semi-Supervised Learning Revisited"](https://arxiv.org/abs/2309.13599)) alternates two closed-form updates instead of backpropagating through a deep GNN — a **lazy graph convolution (LGC)** smoothing step that lazily random-walks node embeddings `U` toward their neighbors, and a **supervised embedding bump (SEB)** step that nudges `U` using the error of a simple linear classifier `W` refit on the labeled nodes each iteration. This notebook ports the reference implementation to K3-Node: `k3_node.transforms.GCNNorm` builds the symmetric-normalized adjacency, `K3LinearNeuralNetwork` is the linear classifier `W`, and both update rules are plain `keras.ops` linear algebra — since neither step needs autodiff, the whole loop runs identically on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install git+http://github.com/anas-rz/k3-node/@main

# ==============================================================================
# K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import numpy as np
import keras
from keras import layers, ops

from k3_node.datasets import Planetoid
from k3_node import transforms as k3_transforms

title = "OGC: From Cluster Assumption to Graph Convolution"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# Hyperparameters from the paper (https://arxiv.org/abs/2309.13599)
decline = 0.9        # decline rate for the supervised learning rate
eta_sup = 0.001      # learning rate for the supervised (SEB) loss
eta_W = 0.5          # learning rate for updating the linear classifier W
beta = 0.1           # lazy random-walk probability of moving to a neighbor
max_sim_tol = 0.995  # max prediction similarity between consecutive iterations
max_patience = 2     # tolerance for consecutive near-identical test predictions

# 1. Dataset: Cora with GCN-normalized (symmetric, self-looped) edge weights
transform = k3_transforms.Compose([k3_transforms.NormalizeFeatures(), k3_transforms.GCNNorm()])
dataset = Planetoid(root="./data/Planetoid", name="Cora", transform=transform)
data = dataset[0]
num_nodes = data.num_nodes
num_features = dataset.num_features
num_classes = dataset.num_classes

# 2. Dense lazy-random-walk adjacency matrix. Cora is small enough that a
# dense N x N matrix is simplest and keeps every op below backend-agnostic.
edge_index_np = ops.convert_to_numpy(data.edge_index)
edge_weight_np = ops.convert_to_numpy(data.edge_weight)
adj_norm = np.zeros((num_nodes, num_nodes), dtype=np.float32)
adj_norm[edge_index_np[0], edge_index_np[1]] = edge_weight_np
lazy_adj = ops.convert_to_tensor(beta * adj_norm + (1 - beta) * np.eye(num_nodes, dtype=np.float32))

y_one_hot = ops.one_hot(ops.cast(data.y, "int32"), num_classes)

train_mask_np = ops.convert_to_numpy(data.train_mask).astype(bool)
val_mask_np = ops.convert_to_numpy(data.val_mask).astype(bool)
test_mask_np = ops.convert_to_numpy(data.test_mask).astype(bool)
trainval_idx = np.nonzero(train_mask_np | val_mask_np)[0]
test_idx = np.nonzero(test_mask_np)[0]
# S = diag(train_mask), used as an elementwise row-mask instead of a matmul
train_mask_col = ops.convert_to_tensor(train_mask_np.astype("float32").reshape(-1, 1))


# 3. Linear classifier W. Since it's just least-squares regression, `update_W`
# solves a single (momentum-free) SGD step in closed form -- identical to
# `loss.backward(); optimizer.step()` -- so no autodiff/backend branching
# is needed anywhere in this notebook.
class K3LinearNeuralNetwork(keras.Model):
    def __init__(self, num_features, num_classes):
        super().__init__()
        self.W = layers.Dense(num_classes, use_bias=False)
        self.W.build((None, num_features))

    def call(self, x):
        return self.W(x)

    def test(self, U, y_one_hot):
        out = self(U)
        trainval_pred = ops.take(out, trainval_idx, axis=0)
        trainval_y = ops.take(y_one_hot, trainval_idx, axis=0)
        loss = ops.mean(ops.square(trainval_pred - trainval_y))

        pred = ops.argmax(out, axis=-1)
        y_true = ops.argmax(y_one_hot, axis=-1)
        trainval_acc = ops.mean(ops.cast(
            ops.take(pred, trainval_idx, axis=0) == ops.take(y_true, trainval_idx, axis=0), "float32"))
        test_acc = ops.mean(ops.cast(
            ops.take(pred, test_idx, axis=0) == ops.take(y_true, test_idx, axis=0), "float32"))
        return (
            float(ops.convert_to_numpy(loss)),
            float(ops.convert_to_numpy(trainval_acc)),
            float(ops.convert_to_numpy(test_acc)),
            pred,
        )

    def update_W(self, U, y_one_hot, eta_W):
        U_masked = ops.take(U, trainval_idx, axis=0)
        y_masked = ops.take(y_one_hot, trainval_idx, axis=0)
        residual = ops.matmul(U_masked, self.W.kernel) - y_masked
        grad = 2 * ops.matmul(ops.transpose(U_masked), residual)
        self.W.kernel.assign(self.W.kernel - eta_W * grad)
        return self(U), self.W.kernel


model = K3LinearNeuralNetwork(num_features, num_classes)


def update_U(U, pred, W, eta_sup):
    # Smoothness update via lazy graph convolution (LGC)
    U = ops.matmul(lazy_adj, U)
    # Supervised error-correction update (SEB), restricted to train_mask
    diff = (pred - y_one_hot) * train_mask_col
    dU_sup = 2 * ops.matmul(diff, ops.transpose(W))
    return U - eta_sup * dU_sup


def ogc():
    global eta_sup
    U = data.x
    _, _, last_acc, last_pred = model.test(U, y_one_hot)

    patience = 0
    for i in range(1, 21):
        pred, W = model.update_W(U, y_one_hot, eta_W)
        U = update_U(U, pred, W, eta_sup)
        eta_sup = eta_sup * decline

        loss, trainval_acc, test_acc, pred_labels = model.test(U, y_one_hot)
        print(f"Epoch: {i:02d}, Loss: {loss:.4f}, "
              f"Train+Val Acc: {trainval_acc:.4f} Test Acc: {test_acc:.4f}")

        sim_rate = float(ops.convert_to_numpy(ops.mean(ops.cast(pred_labels == last_pred, "float32"))))
        if sim_rate > max_sim_tol:
            patience += 1
            if patience > max_patience:
                break

        last_acc, last_pred = test_acc, pred_labels

    return last_acc


print(f"Running OGC on {backend} backend...")
test_acc = ogc()
print(f"Test Accuracy: {test_acc:.4f}")

print("\n✓ K3-Node execution completed successfully!")
